## Import libraries

In [1]:
import os
import requests
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torchvision
from torchvision import models, transforms
from torchsummary import summary
from pinecone import Pinecone, ServerlessSpec
from dotenv import load_dotenv, dotenv_values 

load_dotenv()

INDEX_DIMENSION=1024 # defined based on convnext "base" model setting for embeggings
PINECONE_KEY=os.getenv("PINECONE_KEY")
INDEX_NAME=os.getenv("INDEX_NAME")


/Users/mariiatorubara/masters/.venv/lib/python3.10/site-packages/pinecone/data/index.py:1: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


## EDA (tbd)

Load .csv, .json and images data

In [54]:
styles_metadata = pd.read_csv("../fashion-dataset/styles.csv", delimiter=';').set_index("id")
images_metadata = pd.read_csv("../fashion-dataset/images.csv", delimiter=';')

In [20]:
image_ids = np.sort(styles_metadata.id)
image_address = ["../fashion-dataset/images/"+str(i)+".jpg" for i in image_ids]
image_address_dictionary = {int(k):v for k,v in zip(image_ids,image_address )}

In [9]:
image_set = {}
missed_items_ids = []
for id in image_address_dictionary:
    address = image_address_dictionary[id]
    try:
        image_set[id] = Image.open(address)
    except:
        missed_items_ids.append(id)

# 1. Create Pinecone Index

In [62]:
# initialize Pinecone
pc = Pinecone(api_key=PINECONE_KEY)

In [5]:
# create index
if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME, 
        dimension=INDEX_DIMENSION*2,
        spec=ServerlessSpec(
            cloud='aws',
            region='us-east-1'
        )
    )

index = pc.Index(INDEX_NAME)


In [66]:
# create index
if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME, 
        dimension=INDEX_DIMENSION,
        spec=ServerlessSpec(
            cloud='aws',
            region='us-east-1'
        )
    )

index = pc.Index(INDEX_NAME)

In [67]:
index

# 2. Prepare embeddings

Download pre-trained model:

In [7]:
# loading pretrained ConvNeXT
# link to ConvNeXt Base weights
#url = "https://download.pytorch.org/models/convnext_base-6075fbad.pth"

# download the weights
#response = requests.get(url)
#with open("convnext_base.pth", "wb") as f:
#    f.write(response.content)

# load the model with the downloaded weights
model = models.convnext_base()
model.load_state_dict(torch.load("convnext_base.pth"))


/var/folders/73/8x9p3dg55zz90b9r5c3w9nlh0000gn/T/ipykernel_13802/3012632641.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("convnext_b

<All keys matched successfully>

Model summary:

In [8]:
channels = 3 # image has 3 channels -> RGB 
height = 224 # pixels 
width = 224 # pixels
input_size = (channels, height, width)
summary(model, input_size)

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1          [-1, 128, 56, 56]           6,272
       LayerNorm2d-2          [-1, 128, 56, 56]             256
            Conv2d-3          [-1, 128, 56, 56]           6,400
           Permute-4          [-1, 56, 56, 128]               0
         LayerNorm-5          [-1, 56, 56, 128]             256
            Linear-6          [-1, 56, 56, 512]          66,048
              GELU-7          [-1, 56, 56, 512]               0
            Linear-8          [-1, 56, 56, 128]          65,664
           Permute-9          [-1, 128, 56, 56]               0
  StochasticDepth-10          [-1, 128, 56, 56]               0
          CNBlock-11          [-1, 128, 56, 56]               0
           Conv2d-12          [-1, 128, 56, 56]           6,400
          Permute-13          [-1, 56, 56, 128]               0
        LayerNorm-14          [-1, 56, 

In [9]:
# remove final layer of the model; model.children() aka layers
layers_without_classifier = list(model.children())[:-1]
# unpack layers and init feature exractor
feature_extractor = torch.nn.Sequential(*layers_without_classifier)
# set to evaluation mode
feature_extractor.eval()

Sequential(
  (0): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 128, kernel_size=(4, 4), stride=(4, 4))
      (1): LayerNorm2d((128,), eps=1e-06, elementwise_affine=True)
    )
    (1): Sequential(
      (0): CNBlock(
        (block): Sequential(
          (0): Conv2d(128, 128, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=128)
          (1): Permute()
          (2): LayerNorm((128,), eps=1e-06, elementwise_affine=True)
          (3): Linear(in_features=128, out_features=512, bias=True)
          (4): GELU(approximate='none')
          (5): Linear(in_features=512, out_features=128, bias=True)
          (6): Permute()
        )
        (stochastic_depth): StochasticDepth(p=0.0, mode=row)
      )
      (1): CNBlock(
        (block): Sequential(
          (0): Conv2d(128, 128, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=128)
          (1): Permute()
          (2): LayerNorm((128,), eps=1e-06, elementwise_affine=True)
          (3): Linear(in_fe

In [10]:
# prepare images for embeddings creation
preprocess = transforms.Compose([ 
    transforms.Resize(height+32), # crop the shortes side to 256 pixels (height+32 pixels margin)
    transforms.CenterCrop(height), # center and cop image to 224x2245 soize
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], # average pixel values for Red, Green, and Blue channels in the ImageNet dataset
        std=[0.229, 0.224, 0.225]), # e standard deviation of pixel values for each channel in ImageNet
])

In [11]:
# load and preprocess an image
image_path = image_address[0]
image = Image.open(image_path)
input_tensor = preprocess(image).unsqueeze(0)  # add batch dimension

In [12]:
#with torch.no_grad():
#    embedding = feature_extractor(input_tensor)
embedding = feature_extractor(input_tensor)

# Flatten the embedding to get a 1D vector
embedding_vector = embedding.flatten()

# Print the embedding vector
print(embedding_vector.shape)

torch.Size([1024])


do it in batches, orchestrating solution (pipeline script), solution should be (maybe) run in parallel.

for-loop orchestrating solution + log file (if the job was interrupted),

pipline script (could be bash or python) to enable upserating data.

Solution design:

- models of the setup (per logical step: pre-processing, index upserting, training, evaluation, etc.) -> can be a separate folder with sets of the script / or .py function file

pipeline script:
- (1) define chunks (range of indecies) - script 1 - this is an orchestrator 
- (2) preprocess data - script 2 - runs in loop what is required
- (3) script 1 calls script 2


config file: stores params for script executtion (can be used for running too)

In [26]:
embedding_vectors = {}
for image_id in image_ids[:20]:
    # load and preprocess an image
    image_path = image_address_dictionary[image_id]
    image = Image.open(image_path)
    # add batch dimension
    input_tensor = preprocess(image).unsqueeze(0) 
    with torch.no_grad():
        embedding = feature_extractor(input_tensor)
    # flatten the embedding to get a 1D vector
    embedding_vectors[int(image_id)] = embedding.flatten()


## Upserting vectors to the index

In [60]:
vectors_to_upsert = []
metadata_dictionary = styles_metadata.T.to_dict()
for k,v in embedding_vectors.items():
    tmp_dict = {
        "id": str(k),
        "values": v,
        "metadata": metadata_dictionary[k]
    }
    vectors_to_upsert.append(tmp_dict)

In [68]:
index.upsert(vectors=vectors_to_upsert)

{'upserted_count': 20}